In [94]:
import kagglehub
import os
# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")
# Load into a DataFrame
csv_file = os.path.join(path, "WA_Fn-UseC_-Telco-Customer-Churn.csv")

Using Colab cache for faster access to the 'telco-customer-churn' dataset.


# Libraries


In [95]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (OneHotEncoder,StandardScaler)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import(r2_score,mean_squared_error,root_mean_squared_error)

# Dataset


In [96]:
df=pd.read_csv(csv_file)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# Data Inspection


In [97]:
print("Shape: ",df.shape)
print("Nunique: ",df.nunique())

Shape:  (7043, 21)
Nunique:  customerID          7043
gender                 2
SeniorCitizen          2
Partner                2
Dependents             2
tenure                73
PhoneService           2
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
DeviceProtection       3
TechSupport            3
StreamingTV            3
StreamingMovies        3
Contract               3
PaperlessBilling       2
PaymentMethod          4
MonthlyCharges      1585
TotalCharges        6531
Churn                  2
dtype: int64


In [98]:
df.isnull().sum().sort_values(ascending=False)

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [99]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [100]:
# By seeing data we can conclude that Montly Charge ans Total Charge are in object so convvert it to float
df['MonthlyCharges']=pd.to_numeric(df['MonthlyCharges'],errors='coerce')
df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')

In [101]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


# Remove Unnecessary Columns


In [102]:
df.drop(columns=['customerID','Partner'],axis=1,inplace=True)

X / Y seperation

In [103]:
X=df.drop(columns=['Churn'])
y=df['Churn']

Train test split

In [104]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
print(X_train.shape,X_test.shape)

(5634, 18) (1409, 18)


In [105]:
Numeric_Columns=X_train.select_dtypes(include='number').columns.tolist()
Categorical_Columns=X_train.select_dtypes(include='object').columns.tolist()

Check Skewness of Missing Column


In [106]:
X_train['TotalCharges'].skew() # SO medain is best


np.float64(0.9543007029474576)

In [107]:
numeric_pipeline=Pipeline([
    ('impute',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])

In [108]:
categorical_pipeline=Pipeline([
    ('encode',OneHotEncoder(sparse_output=False,handle_unknown='ignore',drop='first'))
])

In [109]:
preprocessing=ColumnTransformer(transformers=[
    ('impute&Scaler',numeric_pipeline,Numeric_Columns),
    ('Encoding',categorical_pipeline,Categorical_Columns)
])

In [110]:
model_pipeline=Pipeline([
    ('process',preprocessing),
    ('model',LogisticRegression())
])

In [111]:
model_pipeline.fit(X_train,y_train)
y_pred=model_pipeline.predict(X_test)

In [112]:
y_test

,Churn
185,Yes
2715,No
3825,No
1807,Yes
132,No
...,...
6366,No
315,No
2439,No
5002,No


In [115]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision: ", precision_score(y_test, y_pred, pos_label='Yes'))
print("Recall: ", recall_score(y_test, y_pred, pos_label='Yes'))
print("F1 Score: ", f1_score(y_test, y_pred, pos_label='Yes'))

Accuracy:  0.8225691980127751
Precision:  0.6892307692307692
Recall:  0.6005361930294906
F1 Score:  0.6418338108882522


# Display Pipeline Architecture


In [116]:
from sklearn import set_config
set_config(display='diagram')
model_pipeline

Pipeline(steps=[('process',
                 ColumnTransformer(transformers=[('impute&Scaler',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('Encoding',
                                                  Pipeline(steps=[('encode',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['gender', 'Dependents',
                                                   'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod'])])),
                ('model', LogisticRegression())])